In [1]:
import pandas as pd
from tqdm import tqdm

# Dataset

In [2]:
aged = pd.read_csv("../data/pubmed_rct/cav_files/aged.tsv", header=None, sep="\t", names=["pmid", "abstract"])
aged["label"] = [1]*len(aged)

aged_flip = pd.read_csv("../data/pubmed_rct/cav_files/aged_flip.tsv", header=None, sep="\t", names=["pmid", "abstract"])
aged_flip["label"] = [0]*len(aged_flip)

child = pd.read_csv("../data/pubmed_rct/cav_files/child.tsv", header=None, sep="\t", names=["pmid", "abstract"])
child["label"] = [0]*len(child)

child_flip = pd.read_csv("../data/pubmed_rct/cav_files/child_flip.tsv", header=None, sep="\t", names=["pmid", "abstract"])
child_flip["label"] = [1]*len(child_flip)

df = pd.concat([aged, aged_flip, child, child_flip], ignore_index=True)
df

,pmid,abstract,label
0,38587261,BACKGROUND: Patients with severe aortic stenos...,1
1,38892674,(1) Background: Global population aging is cha...,1
2,39150711,IMPORTANCE: Delirium is common among older hos...,1
3,38658827,BACKGROUND: Regular exercise is emphasized for...,1
4,39225274,BACKGROUND: Whether a conservative strategy of...,1
...,...,...,...
95,39107823,AIMS: This work aims to investigate whether vi...,1
96,38499947,OBJECTIVES: To compare the effectiveness of ca...,1
97,38951253,Older adults with Cerebral Palsy (CP) experien...,1
98,38961171,Palatal injections are considered to be one of...,1


# Load Embedding Model & Embed

In [3]:
from sentence_transformers import SentenceTransformer

# load embedding model
embedding_model_path = "BAAI/bge-large-en-v1.5"
embedding_model = SentenceTransformer(embedding_model_path)

In [4]:
embeddings = embedding_model.encode(df.abstract.to_list())
embeddings.shape

(100, 1024)

# Identify CAV

In [5]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, roc_auc_score

## Logistic Regression

In [6]:
X = embeddings  
y = np.asarray(df.label.to_list())

kf = KFold(n_splits=10, shuffle=True, random_state=42)

accs = []
aucs = []
cavs = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Train logistic regression
    clf = LogisticRegression(penalty='l2', solver='liblinear')
    clf.fit(X_train, y_train)

    # Save CAV (normalized coef vector)
    cav = clf.coef_.flatten()
    cav = cav / np.linalg.norm(cav)
    cavs.append(cav)

    # Evaluate using classifier outputs
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]  # probability for class 1

    acc = accuracy_score(y_test, y_pred)
    accs.append(acc)

    if len(np.unique(y)) == 2:
        auc = roc_auc_score(y_test, y_proba)
        aucs.append(auc)
        print(f"Fold {fold + 1}: Accuracy = {acc:.3f}, AUC = {auc:.3f}")
    else:
        print(f"Fold {fold + 1}: Accuracy = {acc:.3f}")

# Summary
print(f"\nAverage Accuracy: {np.mean(accs):.3f} ± {np.std(accs):.3f}")
if aucs:
    print(f"Average AUC: {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")

Fold 1: Accuracy = 0.800, AUC = 1.000
Fold 2: Accuracy = 1.000, AUC = 1.000
Fold 3: Accuracy = 1.000, AUC = 1.000
Fold 4: Accuracy = 1.000, AUC = 1.000
Fold 5: Accuracy = 1.000, AUC = 1.000
Fold 6: Accuracy = 1.000, AUC = 1.000
Fold 7: Accuracy = 1.000, AUC = 1.000
Fold 8: Accuracy = 0.900, AUC = 1.000
Fold 9: Accuracy = 1.000, AUC = 1.000
Fold 10: Accuracy = 0.900, AUC = 1.000

Average Accuracy: 0.960 ± 0.066
Average AUC: 1.000 ± 0.000


In [16]:
X = embeddings 
y = np.asarray(df.label.to_list()) 

# 2. Train linear classifier
clf = LogisticRegression(penalty='l2', solver='liblinear')  # or SVM with linear kernel
clf.fit(embeddings, y)

# 3. Extract CAV: normal vector to the decision boundary
cav_age = clf.coef_.flatten()  # shape: (embedding_dim,)

# Optional: normalize the vector
cav_age = cav_age / np.linalg.norm(cav_age)
cav_age.shape

(1024,)

# SVM with Linear Kernel

In [6]:
X = embeddings  
y = np.asarray(df.label.to_list()) 

kf = KFold(n_splits=10, shuffle=True, random_state=42)

accs = []
aucs = []
cavs = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Train logistic regression
    clf = LinearSVC()
    clf.fit(X_train, y_train)

    # Save CAV (normalized coef vector)
    cav = clf.coef_.flatten()
    cav = cav / np.linalg.norm(cav)
    cavs.append(cav)

    # Evaluate using classifier outputs
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    accs.append(acc)

    print(f"Fold {fold + 1}: Accuracy = {acc:.3f}")
    
# Summary
print(f"\nAverage Accuracy: {np.mean(accs):.3f} ± {np.std(accs):.3f}")

Fold 1: Accuracy = 0.800
Fold 2: Accuracy = 1.000
Fold 3: Accuracy = 0.800
Fold 4: Accuracy = 1.000
Fold 5: Accuracy = 1.000
Fold 6: Accuracy = 1.000
Fold 7: Accuracy = 1.000
Fold 8: Accuracy = 0.900
Fold 9: Accuracy = 1.000
Fold 10: Accuracy = 1.000

Average Accuracy: 0.950 ± 0.081


In [7]:
X = embeddings 
y = np.asarray(df.label.to_list())

# 2. Train linear classifier
clf = LinearSVC()
clf.fit(X, y)

# 3. Extract CAV: normal vector to the decision boundary
cav_age = clf.coef_.flatten()  # shape: (embedding_dim,)

# Optional: normalize the vector
cav_age = cav_age / np.linalg.norm(cav_age)
cav_age.shape

(1024,)

# Load ctELM

In [8]:
from src.model import LlamaForEmbeddingLM
from peft import PeftModel
from transformers import AutoTokenizer
import torch
from src.utils import batch_inference

# load tokenizer & elm
model_path = "initial_elm_model"
device = 'cuda'

tokenizer = AutoTokenizer.from_pretrained(model_path)

elm = LlamaForEmbeddingLM.from_pretrained(
    model_path, 
    torch_dtype=torch.bfloat16,
    device_map=device)

#peft_model_id = "full_tuning_lora_outputs/checkpoint-17873"
peft_model_id = "5tasks_full_tuning_lora_outputs/checkpoint-37780"
lora_elm = PeftModel.from_pretrained(elm, peft_model_id)
lora_elm = lora_elm.merge_and_unload()

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [9]:
all_aged = pd.concat([aged, child_flip], ignore_index=True)
all_aged.shape

(50, 3)

In [10]:
# test block
alpha = 2
for _, row in all_aged.iterrows():
    emb = embedding_model.encode(row["abstract"])
    cav_emb = emb + (alpha*cav_age)
    print(np.linalg.norm(cav_emb))
    normalized_cav_emb = cav_emb / np.linalg.norm(cav_emb)
    print(np.linalg.norm(normalized_cav_emb), normalized_cav_emb.shape)

    if _ == 3:
        break

2.296829473286767
1.0 (1024,)
2.362235934086863
1.0 (1024,)
2.366342811790332
1.0 (1024,)
2.361874447565037
0.9999999999999999 (1024,)


In [11]:
pbar = tqdm(total=len(all_aged))
# negative direction is to child
alpha_list = [-0.0625, -0.125, -0.25, -0.5, -1, -2, -4, -8]
for _, row in all_aged.iterrows():
    emb = embedding_model.encode(row["abstract"])

    # create test inputs
    test_embs = []
    for alpha in alpha_list:
        cav_emb = emb + (alpha*cav_age)
        normalized_cav_emb = cav_emb / np.linalg.norm(cav_emb)
        test_embs.append(normalized_cav_emb)
    
    outputs = batch_inference(lora_elm, tokenizer, test_embs,  device="cuda", task="abstract", repetition_penalty=1.2)

    # parse outputs & update the row
    for idx in range(0, len(alpha_list)):
        all_aged.loc[_, "abstract_" + str(alpha_list[idx])] = outputs[idx]
    
    pbar.update(1)
pbar.close()

100%|██████████| 50/50 [06:07<00:00,  7.35s/it]


In [12]:
all_aged.to_csv("../data/pubmed_rct/cav_files/penalty=1.2/all_aged_results.tsv", sep="\t", index=False)

In [13]:
all_aged["abstract_-0.125"].to_list()

["The optimal valve size for transcatheter aortic valve replacement (TAVR) in patients with severe, symptomatic aortic stenosis and small annuli is unknown. We compared outcomes after TAVR using either an Edwards Sapien XT or CoreValve prosthesis according to preprocedural left ventricular outflow tract area. The PARTNER trial randomized 699 high-risk surgical candidates to receive either TAVR or standard therapy; 100 were assigned to undergo TAVR without randomization because they had small annuli (<20 mm). Of these nonrandomized patients, 55 received the Edwards device and 45 received the Medtronic CoreValve. Patients who underwent TAVR with the smaller prostheses had significantly larger mean left ventricular outflow tracts than those receiving the larger devices (2.0+/-1.5 cm vs. 1.6+/-1.7 cm, P=0.02), but similar baseline characteristics otherwise. At discharge, there was no difference between groups regarding mortality (Edwards 10% [n=5] vs. CoreValve 16% [n=8], P=NS); stroke/TIA

In [19]:
all_child = pd.concat([child, aged_flip], ignore_index=True)
all_child.shape

(50, 3)

In [23]:
pbar = tqdm(total=len(all_child))
# positive direction is to aged
alpha_list = [0.0625, 0.125, 0.25, 0.5, 1, 2, 4, 8]
for _, row in all_child.iterrows():
    emb = embedding_model.encode(row["abstract"])

    # create test inputs
    test_embs = []
    for alpha in alpha_list:
        cav_emb = emb + (alpha*cav_age)
        normalized_cav_emb = cav_emb / np.linalg.norm(cav_emb)
        test_embs.append(normalized_cav_emb)
    
    outputs = batch_inference(lora_elm, tokenizer, test_embs, device="cuda", task="abstract", repetition_penalty=1.2)

    # parse outputs & update the row
    for idx in range(0, len(alpha_list)):
        all_child.loc[_, "abstract_" + str(alpha_list[idx])] = outputs[idx]
    
    pbar.update(1)
pbar.close()

100%|██████████| 50/50 [05:29<00:00,  6.60s/it]


In [24]:
all_child.to_csv("../data/pubmed_rct/cav_files/penalty=1.2/all_child_results.tsv", sep="\t", index=False)

In [26]:
all_child["abstract_0.25"].to_list()

['To evaluate the effectiveness and safety of a robot-assisted gait training system (Lokomat) in elderly patients with stroke. A randomized controlled trial was conducted at an acute rehabilitation hospital from January to June, 2011. The study included 30 subjects who were randomly assigned into two groups; Lokomat group (n=15), which received robotic therapy for 20 minutes per day, five days a week during their stay on the ward, or control group (n=15). Outcome measures including walking speed, balance ability, muscle strength, spasticity, functional independence measure (FIM), Barthel index (BI), and quality of life (QOL) were assessed before treatment and after discharge. In addition, we evaluated the feasibility of using this device by assessing user satisfaction and usability. There were no significant differences between the two groups regarding baseline characteristics except age. After intervention, there were statistically significant improvements in FIM scores (-2.0+/-5.6 vs